In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

# Veriyi yükle
dataset = load_dataset("truehealth/medicationqa")
df = dataset["train"].to_pandas()

# Sütun isimlerini sadeleştir
df = df.rename(columns={"Question": "question", "Answer": "answer"})

# Sadece gerekli sütunları al
df = df[["question", "answer"]]

# Boş kayıtları temizle
df = df.dropna()
df = df[df["question"].str.strip() != ""]
df = df[df["answer"].str.strip() != ""]

print(f"Temiz veri sayısı: {len(df)}")
print(f"\nÖrnek soru: {df['question'].iloc[0]}")
print(f"Örnek cevap: {df['answer'].iloc[0][:200]}...")

# Train / Validation / Test olarak böl (%80 / %10 / %10)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"\nEğitim: {len(train_df)}")
print(f"Doğrulama: {len(val_df)}")
print(f"Test: {len(test_df)}")

# Kaydet
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("\nDosyalar kaydedildi ✓")

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece sentence-transformers -q

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from torch.utils.data import Dataset

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def preprocess_function(df):
    questions = ["Answer this medical question: " + str(q) for q in df["question"]]
    answers = [str(a) for a in df["answer"]]

    model_inputs = tokenizer(
        questions,
        max_length=128,
        truncation=True,
        padding=True
    )

    labels = tokenizer(
        answers,
        max_length=256,
        truncation=True,
        padding=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Dataset sınıfı
class MedQADataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

train_encodings = preprocess_function(train_df)
val_encodings = preprocess_function(val_df)

train_dataset = MedQADataset(train_encodings)
val_dataset = MedQADataset(val_encodings)

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tmodel = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

training_args = TrainingArguments(
    output_dir="./flan_t5_medqa",
    num_train_epochs=10,
    per_device_train_batch_size=4,      # ← 8'den 4'e düşürdük
    per_device_eval_batch_size=4,
    learning_rate=5e-5,                  # ← 3e-4'ten düşürdük
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=False,                          # ← fp16 kapattık
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Flan-T5 eğitimi başlıyor...")
trainer.train()
print("Eğitim tamamlandı ✓")


model.save_pretrained("./flan_t5_final")
tokenizer.save_pretrained("./flan_t5_final")
print("Model kaydedildi ✓")

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

# Veriyi yükle
df = pd.read_csv("train.csv")

# Sentence-BERT modeli (bu fine-tune değil, hazır model)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Tüm soruları encode et
print("Sorular encode ediliyor...")
question_embeddings = embedder.encode(df["question"].tolist(), convert_to_tensor=True)
print("Hazır ✓")

def retrieve_answer(user_question, threshold=0.5):
    query_embedding = embedder.encode(user_question, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score < threshold:
        return {
            "bulunan_soru": None,
            "cevap": "Bu soruya uygun bir cevap bulunamadı. Lütfen sorunuzu farklı şekilde sorun.",
            "benzerlik": best_score
        }

    return {
        "bulunan_soru": df["question"].iloc[best_idx],
        "cevap": df["answer"].iloc[best_idx],
        "benzerlik": best_score
    }

# Test et
test_df = pd.read_csv("test.csv")
for i in range(3):
    soru = test_df["question"].iloc[i]
    gercek = test_df["answer"].iloc[i]
    sonuc = retrieve_answer(soru)

    print(f"Soru: {soru}")
    print(f"En benzer soru: {sonuc['bulunan_soru']}")
    print(f"Benzerlik: {sonuc['benzerlik']:.3f}")
    print(f"Gerçek Cevap: {gercek[:150]}...")
    print(f"Model Cevabı: {sonuc['cevap'][:150]}...")
    print("-" * 60)

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# T5 hazır model yükle (fine-tune yok)
t5_tokenizer = T5Tokenizer.from_pretrained("t5-base")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-base")
t5_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
t5_model = t5_model.to(device)
print("T5 hazır ✓")

def rag_answer(user_question):
    sonuc = retrieve_answer(user_question)

    if sonuc["bulunan_soru"] is None:
        return "Bu soruya uygun bir cevap bulunamadı.", 0.0, None

    context = sonuc["cevap"]
    score = sonuc["benzerlik"]
    bulunan_soru = sonuc["bulunan_soru"]

    # Daha açık bir prompt
    input_text = f"question: {user_question} context: {context[:400]}"

    input_ids = t5_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).input_ids.to(device)

    outputs = t5_model.generate(
        input_ids,
        max_length=200,
        min_length=20,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
    )
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True), score, bulunan_soru

# Test et
try:
    for i in range(3):
        soru = test_df["question"].iloc[i]
        gercek = test_df["answer"].iloc[i]
        cevap, score, bulunan_soru = rag_answer(soru)

        print(f"Soru: {soru}")
        print(f"Benzerlik: {score:.3f}")
        print(f"Gerçek: {gercek[:150]}...")
        print(f"RAG Cevabı: {cevap}")
        print("-" * 60)
except Exception as e:
    print(f"Hata oluştu: {e}")

In [11]:
!pip install gradio -q

In [ ]:
import gradio as gr

def rag_chat(user_question, history):
    cevap, score, bulunan_soru = rag_answer(user_question)

    # Benzerlik skoru arayüzden kaldırıldığı için bilgi değişkenine gerek kalmadı.
    history.append((user_question, cevap))
    return history, bulunan_soru, ""

with gr.Blocks(title="İlaç Bilgilendirme Chatbot — RAG") as demo_rag:
    gr.Markdown("# 💊 İlaç Bilgilendirme Chatbot — RAG (Sentence-BERT + T5)")
    gr.Markdown("İlaçlarla ilgili sorularınızı İngilizce olarak sorun.")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Sohbet Geçmişi",
                height=400
            )
            user_input = gr.Textbox(
                placeholder="İlaç sorunuzu buraya yazın...",
                label="Sorunuz",
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button(
                    "Gönder",
                    variant="primary"
                )
                clear_btn = gr.Button("Temizle")

        with gr.Column(scale=1):
            # Skorlar kutucuğu kaldırıldı
            bulunan_soru_box = gr.Textbox(
                label="Eşleşen Soru",
                lines=3,
                interactive=False
            )

    gr.Examples(
        examples=[
            ["what is ibuprofen used for?"],
            ["what are the side effects of aspirin?"],
            ["how is tetracycline metabolized?"],
            ["what is metformin used for?"]
        ],
        inputs=user_input
    )

    history_state = gr.State([])
    submit_btn.click(
        fn=rag_chat,
        inputs=[user_input, history_state],
        outputs=[
            chatbot,
            bulunan_soru_box,
            user_input
        ]

    ).then(
        lambda h: h,
        history_state,
        history_state
    )
    clear_btn.click(
        lambda: ([], "", ""), # Skorlar çıktısı kaldırıldı
        outputs=[
            chatbot,
            history_state,
            bulunan_soru_box
        ]
    )

demo_rag.launch(share=True)

In [13]:
!pip install evaluate rouge_score -q

In [ ]:
import pandas as pd
from evaluate import load

test_df = pd.read_csv("test.csv")

rouge = load("rouge")
bleu = load("bleu")

predictions = []
references = []

print("Test seti üzerinde tahminler üretiliyor...")

for _, row in test_df.iterrows():
    soru = row["question"]
    gercek_cevap = row["answer"]

    cevap, score, bulunan_soru = rag_answer(soru)

    if score == 0.0:
        predictions.append("no answer found")
    else:
        predictions.append(cevap)

    references.append(gercek_cevap)

print(f"Toplam tahmin: {len(predictions)}")

# ROUGE
rouge_scores = rouge.compute(predictions=predictions, references=references)

# BLEU
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

# Exact Match
exact_match = sum(p.strip().lower() == r.strip().lower()
                  for p, r in zip(predictions, references)) / len(predictions)

# F1
def compute_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens = ref.lower().split()
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

f1_scores = [compute_f1(p, r) for p, r in zip(predictions, references)]
avg_f1 = sum(f1_scores) / len(f1_scores)

print("\n=== RAG Değerlendirme Sonuçları ===")
print(f"ROUGE-1:      {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:      {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:      {rouge_scores['rougeL']:.4f}")
print(f"BLEU:         {bleu_score['bleu']:.4f}")
print(f"Exact Match:  {exact_match:.4f}")
print(f"F1 Score:     {avg_f1:.4f}")

# Kaydet
results_df = pd.DataFrame({
    "question": test_df["question"],
    "reference": references,
    "rag_prediction": predictions
})
results_df.to_csv("rag_results.csv", index=False)
print("\nSonuçlar kaydedildi ✓")